# GEM — Geometric Evolution Maps: Paper 2 Companion

*Written: 2026-05-20 02:31 UTC*

This notebook is the reproducibility companion for **Paper 2: Geometric Evolution Maps** (Henry, 2026b). Every quantitative claim in the paper either runs here or is annotated with the aggregate result alongside the code pattern that generates it.

**What GEM is.** CAZ detection (Paper 1) identifies *where* concept assembly happens — the layer range where a concept undergoes its primary separation event. GEM asks what happens to the concept *direction* during and after assembly. The core finding: concept directions rotate substantially inside their CAZ (mean entry-exit cosine 0.233), then settle into a stable orientation at a characteristic *handoff layer* immediately after assembly ends. Probing at the handoff layer outperforms probing at the separation peak in 259/391 trials (66.2%).

**Dataset.** All results use the `paper_n250` dataset: 23 base models × 17 concepts × 250 contrastive pairs, stored locally at `~/rosetta_data/paper_n250/`.

**No GPU required.** All activations and GEM structures are pre-computed. This notebook loads JSON results and runs analysis and visualisation only.

**[github.com/jamesrahenry/Rosetta](https://github.com/jamesrahenry/Rosetta)** · Henry (2026b)

### Sections

| § | Paper section | What this notebook covers |
|---|--------------|---------------------------|
| 1 | §3 — GEM formalism | EEC computation from scratch; handoff layer definition |
| 2 | §4 — Handoff phenomenon | EEC distribution across pairs; per-concept means |
| 3 | §5.1 — Handoff vs peak validation | Ablation comparison for 3 representative models |
| 4 | §5.2 — Adaptive width rule | Near-final rule logic; when it triggers |
| 5 | §5.3 — Direction-specificity control | 377× median ratio; random-direction baseline |
| 6 | §6 — Scale analysis | Pythia scale ladder EEC; 410M–1B emergence window |
| 7 | §7 — OPT-6.7b and gpt2 | Two documented failure modes and their resolution |

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    _pip("rosetta_tools>=1.3.1")
except subprocess.CalledProcessError:
    _pip("rosetta_tools @ git+https://github.com/jamesrahenry/Rosetta_Tools.git@v1.3.1")

_pip("matplotlib", "numpy", "scipy")

In [ ]:
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# rosetta_tools import — GPU host path first, then dev machine
_rt = Path.home() / "rosetta_tools"
if not _rt.exists():
    _rt = Path.home() / "Source" / "Rosetta_Program" / "rosetta_tools"
sys.path.insert(0, str(_rt))

from rosetta_tools.caz import LayerMetrics, find_caz_regions
from rosetta_tools.gem import (
    build_concept_gem, load_gem, gem_diagnostics, build_gem_node_k1,
)

# Local paper_n250 dataset
PAPER_DATA = Path.home() / "rosetta_data" / "paper_n250"

# Convenience: model_id -> local directory name
def model_slug(model_id: str) -> str:
    return model_id.replace("/", "_").replace("-", "_")

def model_dir(model_id: str) -> Path:
    return PAPER_DATA / model_slug(model_id)

def load_caz(model_id: str, concept: str) -> dict:
    p = model_dir(model_id) / f"caz_{concept}.json"
    with open(p) as f:
        return json.load(f)

ALL_CONCEPTS = [
    "agency", "authorization", "causation", "certainty", "credibility",
    "deception", "exfiltration", "formality", "moral_valence", "negation",
    "plurality", "sarcasm", "sentiment", "specificity", "temporal_order",
    "threat_severity", "urgency",
]

# Primary corpus: 23 base models
PRIMARY_MODELS = [
    "EleutherAI/pythia-70m",
    "EleutherAI/pythia-160m",
    "EleutherAI/pythia-410m",
    "EleutherAI/pythia-1b",
    "EleutherAI/pythia-1.4b",
    "EleutherAI/pythia-2.8b",
    "EleutherAI/pythia-6.9b",
    "EleutherAI/pythia-12b",
    "openai-community/gpt2",
    "openai-community/gpt2-large",
    "openai-community/gpt2-xl",
    "facebook/opt-1.3b",
    "facebook/opt-6.7b",
    "Qwen/Qwen2.5-0.5B",
    "Qwen/Qwen2.5-1.5B",
    "Qwen/Qwen2.5-3B",
    "Qwen/Qwen2.5-7B",
    "Qwen/Qwen2.5-14B",
    "mistralai/Mistral-7B-v0.3",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "microsoft/phi-2",
]

MHA_MODELS = [
    "EleutherAI/pythia-70m", "EleutherAI/pythia-160m",
    "EleutherAI/pythia-410m", "EleutherAI/pythia-1b",
    "EleutherAI/pythia-1.4b", "EleutherAI/pythia-2.8b",
    "EleutherAI/pythia-6.9b", "EleutherAI/pythia-12b",
    "openai-community/gpt2", "openai-community/gpt2-large",
    "openai-community/gpt2-xl",
    "facebook/opt-1.3b", "facebook/opt-6.7b",
]
GQA_MODELS = [
    "Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-1.5B",
    "Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-7B", "Qwen/Qwen2.5-14B",
    "mistralai/Mistral-7B-v0.3", "meta-llama/Llama-3.1-8B",
]

print(f"Dataset: {PAPER_DATA}")
print(f"Primary corpus: {len(PRIMARY_MODELS)} models x {len(ALL_CONCEPTS)} concepts = {len(PRIMARY_MODELS)*len(ALL_CONCEPTS)} pairs")

In [ ]:
try:
    from rosetta_tools.viz_style import concept_color, THEME, apply_theme
except ImportError:
    def concept_color(c):
        palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
        return palette[hash(c) % len(palette)]
    THEME = {"spine": "#aaaaaa"}
    def apply_theme(ax):
        for spine in ax.spines.values():
            spine.set_color(THEME["spine"])

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "figure.dpi": 150,
    "figure.facecolor": "white",
})
print("Style loaded.")

## 1. GEM formalism — EEC and the handoff layer

Paper §3 defines two central quantities:

**Entry-exit cosine (EEC):** The cosine similarity between the dominant concept direction at CAZ entry and at CAZ exit. A value of 1.0 means no rotation; 0.0 means orthogonal directions before and after assembly.

$$\text{EEC} = \left| u^{(L_{\text{start}})} \cdot u^{(L_{\text{end}})} \right|$$

**Handoff layer:** The first post-assembly layer — the layer immediately after the CAZ ends:

$$L_H = \min(L_{\text{CAZ\_end}} + 1,\; N - 1)$$

The settled direction $u^{(L_H)}$ is the GEM-extracted probe: the concept direction after the primary rotation has ceased.

The cell below computes these for three example pairs and confirms the core observation: EEC is far below 1.0 in all cases, meaning the direction at CAZ entry does not predict the direction at exit.

In [ ]:
DEMO_PAIRS = [
    ("EleutherAI/pythia-160m", "causation"),
    ("Qwen/Qwen2.5-7B",        "credibility"),
    ("meta-llama/Llama-3.1-8B","certainty"),
]

print(f"{'Model':<30} {'Concept':<16} {'N':>4} {'CAZ start':>10} {'CAZ end':>8} "
      f"{'L_H':>4} {'EEC':>7} {'Handoff cos':>12}")
print("-" * 100)

for model_id, concept in DEMO_PAIRS:
    caz_data = load_caz(model_id, concept)
    metrics  = caz_data["layer_data"]["metrics"]
    n_layers = caz_data["n_layers"]

    lm = [LayerMetrics(layer=m["layer"], separation=m["separation_fisher"],
                       coherence=m["coherence"], velocity=m["velocity"])
          for m in metrics]
    profile = find_caz_regions(lm)

    if profile.n_regions == 0:
        print(f"{model_id.split('/')[-1]:<30} {concept:<16}  — no CAZ detected")
        continue

    # Use the deepest (last) CAZ region — the one that produces the handoff
    region = profile.regions[-1]

    # EEC: entry vs exit direction within the CAZ
    entry_vec = np.array(metrics[region.start]["dom_vector"], dtype=np.float64)
    exit_vec  = np.array(metrics[region.end  ]["dom_vector"], dtype=np.float64)
    entry_vec /= (np.linalg.norm(entry_vec) + 1e-12)
    exit_vec  /= (np.linalg.norm(exit_vec)  + 1e-12)
    eec = abs(float(np.dot(entry_vec, exit_vec)))

    # Handoff layer and its direction
    L_H = min(region.end + 1, n_layers - 1)
    handoff_vec = np.array(metrics[L_H]["dom_vector"], dtype=np.float64)
    handoff_vec /= (np.linalg.norm(handoff_vec) + 1e-12)

    # Handoff cosine: how stable is the settled direction through remaining depth?
    final_vec = np.array(metrics[-1]["dom_vector"], dtype=np.float64)
    final_vec /= (np.linalg.norm(final_vec) + 1e-12)
    handoff_cos = abs(float(np.dot(handoff_vec, final_vec)))

    label = model_id.split("/")[-1]
    start_pct = region.start / n_layers * 100
    end_pct   = region.end   / n_layers * 100
    print(f"{label:<30} {concept:<16} {n_layers:>4} {start_pct:>9.1f}% {end_pct:>7.1f}%"
          f" {L_H:>4} {eec:>7.3f} {handoff_cos:>12.3f}")

print()
print("EEC << 1.0 in all cases: concept direction at CAZ entry does not predict the settled direction.")
print("Handoff cosine near 1.0: the settled direction is stable through remaining model depth.")
print()
print("Paper reports across 391 pairs:  mean EEC = 0.233  |  mean handoff cosine = 0.942")

The pattern holds across model families. EEC measures how much the direction rotates *during* assembly; the handoff cosine measures how stable the settled direction is *after* assembly. These are independent: a concept can rotate substantially during assembly and then stabilise completely at $L_H$.

The `load_gem` function gives direct access to pre-computed GEM nodes (including EEC, handoff layer, and settled direction) without re-running the builder:

In [ ]:
# Load a pre-computed GEM and inspect its structure
example_model   = "Qwen/Qwen2.5-7B"
example_concept = "credibility"

gem = load_gem(model_dir(example_model) / f"gem_{example_concept}.json")
diag = gem_diagnostics(gem)

print(f"ConceptGEM: {gem.model_id} / {gem.concept}")
print(f"  n_nodes:              {gem.n_nodes}")
print(f"  node_types:           {gem.node_types}")
print(f"  ablation_targets:     {gem.ablation_targets}")
print()
for i, node in enumerate(gem.nodes):
    print(f"  Node {i}: CAZ L{node.caz_start}–{node.caz_end} "
          f"(peak L{node.caz_peak}, score={node.caz_score:.3f})")
    print(f"    handoff_layer={node.handoff_layer} "
          f"({node.depth_pct:.1f}% depth)")
    print(f"    entry_exit_cosine={node.entry_exit_cosine:.4f}")
    print(f"    handoff_cosine={node.handoff_cosine:.4f}")
    print(f"    max_rotation_per_layer={node.max_rotation_per_layer:.4f}")
print()
print("Aggregate diagnostics:")
for k, v in diag.items():
    print(f"  {k}: {v}")

## 2. EEC distribution across the corpus (paper §4)

Paper Table 1 reports: mean EEC = 0.233, median = 0.216, 93.9% of pairs with EEC < 0.5, 23.8% near-orthogonal (EEC < 0.1).

The cell below computes EEC for 3 representative models (Pythia-160M, Pythia-2.8B, Qwen2.5-7B) across all 17 concepts and plots the distribution. The paper aggregate result requires iterating all 23 models; it is stated in the markdown note below.

In [ ]:
DEMO_MODELS = [
    "EleutherAI/pythia-160m",
    "EleutherAI/pythia-2.8b",
    "Qwen/Qwen2.5-7B",
]

def compute_eec_for_model(model_id: str) -> dict:
    """Compute EEC for all 17 concepts in a model.
    Returns dict: concept -> EEC (or None if no CAZ).
    """
    results = {}
    for concept in ALL_CONCEPTS:
        try:
            caz_data = load_caz(model_id, concept)
        except FileNotFoundError:
            results[concept] = None
            continue
        metrics  = caz_data["layer_data"]["metrics"]
        n_layers = caz_data["n_layers"]
        lm = [LayerMetrics(layer=m["layer"], separation=m["separation_fisher"],
                           coherence=m["coherence"], velocity=m["velocity"])
              for m in metrics]
        profile = find_caz_regions(lm)
        if profile.n_regions == 0:
            results[concept] = None
            continue
        region    = profile.regions[-1]
        entry_vec = np.array(metrics[region.start]["dom_vector"], dtype=np.float64)
        exit_vec  = np.array(metrics[region.end  ]["dom_vector"], dtype=np.float64)
        entry_vec /= (np.linalg.norm(entry_vec) + 1e-12)
        exit_vec  /= (np.linalg.norm(exit_vec)  + 1e-12)
        results[concept] = abs(float(np.dot(entry_vec, exit_vec)))
    return results

all_eec = {}  # model_id -> {concept: eec}
for mid in DEMO_MODELS:
    all_eec[mid] = compute_eec_for_model(mid)
    valid = [v for v in all_eec[mid].values() if v is not None]
    print(f"{mid.split('/')[-1]:<20}  mean EEC = {np.mean(valid):.3f}  "
          f"median = {np.median(valid):.3f}  "
          f"% < 0.5 = {100*sum(v<0.5 for v in valid)/len(valid):.1f}%  "
          f"% < 0.1 = {100*sum(v<0.1 for v in valid)/len(valid):.1f}%")

In [ ]:
# Plot EEC distributions for the 3 demo models
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
fig.patch.set_facecolor("white")
fig.suptitle("Entry-Exit Cosine (EEC) distributions — 3 representative models",
             fontsize=12, fontweight="bold")

for ax, mid in zip(axes, DEMO_MODELS):
    vals = [v for v in all_eec[mid].values() if v is not None]
    label = mid.split("/")[-1]
    ax.hist(vals, bins=10, range=(0, 1), color="#1565C0", alpha=0.75, edgecolor="white")
    ax.axvline(np.mean(vals),   color="#C62828", lw=2,   ls="--", label=f"mean {np.mean(vals):.3f}")
    ax.axvline(np.median(vals), color="#E65100", lw=1.5, ls=":",  label=f"median {np.median(vals):.3f}")
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.set_xlabel("EEC", fontsize=10)
    ax.legend(fontsize=8)
    apply_theme(ax)

axes[0].set_ylabel("Concept count (N=17)", fontsize=10)
plt.tight_layout()
plt.show()

print()
print("Paper aggregate (391 pairs, 23 models x 17 concepts):")
print("  Mean EEC:          0.233")
print("  Median EEC:        0.216")
print("  % pairs EEC < 0.5: 93.9%")
print("  % pairs EEC < 0.1: 23.8%")
print("  Mean max rotation per layer: 0.309")

### Per-concept EEC means (paper Table 2, selective)

The paper reports per-concept mean EEC across all 23 models. The cells below compute it for the 3 demo models, allowing comparison with the paper values. Key paper finding: **certainty and threat_severity have the highest EEC** (rotate least within their CAZ) while still having mean EEC well below 0.5.

In [ ]:
# Per-concept EEC across the 3 demo models (mean over models)
concept_means = {}
for concept in ALL_CONCEPTS:
    vals = [all_eec[mid][concept] for mid in DEMO_MODELS if all_eec[mid].get(concept) is not None]
    if vals:
        concept_means[concept] = np.mean(vals)

sorted_concepts = sorted(concept_means, key=lambda c: concept_means[c])

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor("white")

xs = range(len(sorted_concepts))
colors = [concept_color(c) for c in sorted_concepts]
ax.bar(xs, [concept_means[c] for c in sorted_concepts], color=colors, alpha=0.85)
ax.axhline(0.5, color="#C62828", lw=1.5, ls="--", alpha=0.7, label="EEC = 0.5")
ax.set_xticks(list(xs))
ax.set_xticklabels([c.replace("_", "\n") for c in sorted_concepts], fontsize=8)
ax.set_ylabel("Mean EEC (3 demo models)", fontsize=10)
ax.set_title("Per-concept EEC — sorted ascending (lower = more rotation during assembly)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
apply_theme(ax)
plt.tight_layout()
plt.show()

print("Paper: all 17 concepts have mean EEC < 0.5 across all 23 models.")
print("Paper: certainty and threat_severity show highest EEC (rotate least during assembly).")
print("Paper per-model EEC range: gpt2 = 0.083 (most rotation) → Qwen2.5-1.5B = 0.392 (least)")

## 3. Handoff vs peak validation — ablation experiment (paper §5.1)

The paper's main validation: for each concept × model pair, compare the GEM-extracted probe (at $L_H$, the handoff layer) against the peak-layer probe (at $L_{\text{peak}}$, the layer of maximum separation). The metric is **retained percentage after ablation** — the fraction of concept separation remaining after projecting out the probe direction. Lower retained = stronger probe.

A **handoff improvement** is when the handoff probe achieves lower retained percentage than the peak probe.

**Paper result:** 259/391 pairs (66.2%) prefer handoff; Wilcoxon signed-rank p = 3.21 × 10⁻¹⁷ (trial-level); model-level W=214, N=23, p=0.010.

The cells below show the comparison for 3 representative models. The pre-computed `ablation_gem_*.json` files contain both handoff and peak comparison data.

In [ ]:
ABLATION_DEMO_MODELS = [
    ("EleutherAI/pythia-2.8b",  "Pythia-2.8B",  "MHA"),
    ("Qwen/Qwen2.5-7B",          "Qwen2.5-7B",   "GQA"),
    ("meta-llama/Llama-3.1-8B",  "Llama-3.1-8B", "GQA"),
]

def load_ablation_gem(model_id: str, concept: str) -> dict:
    p = model_dir(model_id) / f"ablation_gem_{concept}.json"
    with open(p) as f:
        return json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor("white")
fig.suptitle("Handoff vs peak ablation: retained % per concept (lower = better probe)",
             fontsize=12, fontweight="bold")

for ax, (mid, label, arch) in zip(axes, ABLATION_DEMO_MODELS):
    handoff_vals, peak_vals, concepts_loaded = [], [], []
    better_count = 0

    for concept in ALL_CONCEPTS:
        try:
            abl = load_ablation_gem(mid, concept)
        except FileNotFoundError:
            continue
        comp = abl["comparison"]
        handoff_vals.append(comp["handoff_retained_pct"])
        peak_vals.append(comp["peak_retained_pct"])
        concepts_loaded.append(concept)
        if comp["handoff_better"]:
            better_count += 1

    xs = range(len(concepts_loaded))
    ax.scatter(xs, handoff_vals, s=50, color="#1565C0", label="Handoff", zorder=3, alpha=0.9)
    ax.scatter(xs, peak_vals,    s=50, color="#E65100", label="Peak",    zorder=3, alpha=0.9, marker="^")
    for i in range(len(xs)):
        ax.plot([i, i], [handoff_vals[i], peak_vals[i]],
                color="#aaaaaa", lw=0.8, zorder=2)

    ax.set_title(f"{label} ({arch})\n{better_count}/{len(concepts_loaded)} handoff better",
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("Concept index", fontsize=9)
    ax.set_ylabel("Retained %", fontsize=9)
    ax.legend(fontsize=8)
    apply_theme(ax)

plt.tight_layout()
plt.show()

print("Paper aggregate across 391 pairs (23 models x 17 concepts):")
print("  259/391 (66.2%) prefer handoff layer")
print("  Wilcoxon signed-rank (trial-level): p = 3.21e-17")
print("  Model-level Wilcoxon W=214, N=23: p = 0.010 (excl. gpt2: p = 0.0036)")
print("  Llama-3.1-8B: 17/17 = 100%; Pythia-1.4B: 17/17 = 100%")
print("  gpt2: structured failure (13/17 pathological increase — see §7)")

The architecture split is pronounced: MHA models (Pythia, GPT-2, OPT) favour handoff in 11/13 models; GQA models (Qwen, Mistral, Llama) favour handoff in only 2/7 models. Fisher's exact test, one-sided p = 0.022.

Scale breakdown (paper §5.1):

| Scale | Models | Handoff improvement rate |
|-------|--------|-------------------------|
| < 500M (excl. gpt2) | 3 | 61% (31/51) |
| 500M–3B | 11 | 71% (132/187) |
| > 3B | 8 | 70% (95/136) |

The gap between <500M and larger models reflects genuine scale effects, not only the gpt2 artifact.

In [ ]:
# Compute per-model handoff improvement rates for the 3 demo models
print(f"{'Model':<25} {'Arch':<5} {'Better/Total':>12} {'Rate':>7} {'Mean handoff ret%':>18} {'Mean peak ret%':>15}")
print("-" * 85)

for mid, label, arch in ABLATION_DEMO_MODELS:
    better, total = 0, 0
    handoff_rets, peak_rets = [], []
    for concept in ALL_CONCEPTS:
        try:
            abl = load_ablation_gem(mid, concept)
        except FileNotFoundError:
            continue
        comp = abl["comparison"]
        total += 1
        if comp["handoff_better"]:
            better += 1
        handoff_rets.append(comp["handoff_retained_pct"])
        peak_rets.append(comp["peak_retained_pct"])

    rate = better / total * 100 if total else 0
    print(f"{label:<25} {arch:<5} {better:>5}/{total:<6} {rate:>6.1f}%"
          f" {np.mean(handoff_rets):>17.1f}% {np.mean(peak_rets):>14.1f}%")

print()
print("Net expected improvement per pair (paper):")
print("  improvement_mean × rate − degradation_mean × (1−rate)")
print("  = 20.4 × 0.662 − 16.9 × 0.338 = +7.8pp  (matches observed +7.78pp)")

## 4. Adaptive ablation width rule (paper §5.2)

When evaluating probes via ablation, the default ablation width is $w = 3$ consecutive layers starting at $L_H$. For near-final-layer handoffs ($L_H / N > 0.85$), ablating 3 layers extends into the unembedding-preparation zone, contaminating the measurement.

**Near-final rule:** when $L_H / N > 0.85$, set $w = 1$.

**Paper result (32-model, 544-pair extended corpus):** The rule triggers in 89/544 cases (16.4%). When triggered, it improves probe quality in 62/89 cases (70%) with mean improvement +8.51pp. Overall mean improvement across all 544 pairs: +0.69pp.

The cell below identifies which concepts trigger the near-final rule for 3 representative models.

In [ ]:
NEAR_FINAL_THRESHOLD = 0.85

print("Near-final rule triggers (L_H / N > 0.85)")
print("="*60)

for mid in ["EleutherAI/pythia-12b", "Qwen/Qwen2.5-14B", "meta-llama/Llama-3.1-8B"]:
    label = mid.split("/")[-1]
    triggers = []
    for concept in ALL_CONCEPTS:
        try:
            caz_data = load_caz(mid, concept)
        except FileNotFoundError:
            continue
        metrics  = caz_data["layer_data"]["metrics"]
        n_layers = caz_data["n_layers"]
        lm = [LayerMetrics(layer=m["layer"], separation=m["separation_fisher"],
                           coherence=m["coherence"], velocity=m["velocity"])
              for m in metrics]
        profile = find_caz_regions(lm)
        if profile.n_regions == 0:
            continue
        region = profile.regions[-1]
        L_H = min(region.end + 1, n_layers - 1)
        rel_depth = L_H / n_layers
        if rel_depth > NEAR_FINAL_THRESHOLD:
            triggers.append((concept, L_H, n_layers, rel_depth))

    print(f"\n{label} (N={n_layers} layers): {len(triggers)}/17 concepts trigger near-final rule")
    for concept, L_H, N, rel in triggers:
        w_adaptive = 1 if rel > NEAR_FINAL_THRESHOLD else 3
        print(f"  {concept:<18} L_H={L_H:>3}  depth={rel:.3f}  w={w_adaptive}")

print()
print("Depth-corrected rule (recommended): HL/N > 0.85 AND N >= 20")
print("Excludes gpt2 (N=12), pythia-1b (N=16), Llama-3.2-1B (N=16)")
print()
print("Paper results on 32-model extended corpus (544 pairs):")
print("  Triggered: 89 cases  |  Improved: 62/89 (70%)  |  Mean delta +8.51pp (improved cases)")
print("  Overall mean delta across all 544 pairs: +0.69pp")

The adaptive rule is an architectural correction, not a post-hoc optimisation. Models with many layers (Pythia-12b, large Qwen variants) naturally produce more near-final handoffs because their angular velocity threshold is satisfied later in the network as signals become more stable. The rule prevents those legitimate handoffs from being contaminated by unembedding computations.

**gpt2 failure** (§5.2 and §7): gpt2 has N=12 layers; its handoffs typically fall at layer 11 (HL/N ≈ 0.92), where even width-1 ablation engages unembedding-preparation. The depth-corrected rule (N ≥ 20) excludes gpt2 entirely.

## 5. Direction-specificity control (paper §5.3)

A natural sceptical question: does ablating *any* direction suppress concept separation, or is the effect specific to the GEM-extracted concept direction?

The paper's control: for each concept × model pair with non-zero concept-direction reduction, sample 10 random unit vectors and measure the resulting separation reduction. Compare the concept direction's reduction against the random distribution.

**Paper result across 111 pairs (16 models):**
- Mean concept-direction reduction: 45.6%
- Mean random-direction reduction: 0.24%
- Median specificity ratio: 377×
- Pairs where concept > all 10 random seeds: 99.1% (110/111)

The cell below computes this for 3 models using the pre-computed `ablation_random_*.json` files.

In [ ]:
def load_ablation_random(model_id: str, concept: str) -> dict | None:
    p = model_dir(model_id) / f"ablation_random_{concept}.json"
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)

RANDOM_DEMO_MODELS = [
    "EleutherAI/pythia-2.8b",
    "Qwen/Qwen2.5-7B",
    "google/gemma-2-9b",
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor("white")
fig.suptitle("Direction-specificity control: concept vs random ablation reduction",
             fontsize=12, fontweight="bold")

for ax, mid in zip(axes, RANDOM_DEMO_MODELS):
    label = mid.split("/")[-1]
    concept_reds, random_means, ratios = [], [], []

    for concept in ALL_CONCEPTS:
        rdata = load_ablation_random(mid, concept)
        if rdata is None or rdata.get("concept_direction_reduction", 0) <= 0:
            continue
        concept_red = rdata["concept_direction_reduction"]
        rand_mean   = rdata["random_mean_reduction"]
        ratio       = rdata["specificity_ratio"]
        concept_reds.append(concept_red)
        random_means.append(rand_mean)
        ratios.append(ratio)

    if not concept_reds:
        ax.text(0.5, 0.5, "No random ablation data",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_title(label, fontsize=10)
        continue

    xs = range(len(concept_reds))
    ax.bar(xs, [c * 100 for c in concept_reds],
           color="#1565C0", alpha=0.8, label="Concept direction", width=0.8)
    ax.bar(xs, [r * 100 for r in random_means],
           color="#E65100", alpha=0.9, label="Random mean", width=0.8)

    ax.set_title(f"{label}\nmedian ratio {np.median(ratios):.0f}×  (N={len(concept_reds)})",
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("Concept index", fontsize=9)
    ax.set_ylabel("Sep. reduction %", fontsize=9)
    ax.legend(fontsize=8)
    apply_theme(ax)

plt.tight_layout()
plt.show()

print("Paper aggregate across 111 pairs (16 models):")
print("  Mean concept-direction reduction:   45.6%")
print("  Mean random-direction reduction:     0.24%")
print("  Median specificity ratio:            377x")
print("  Median z-score:                      252.7")
print("  Pairs > all 10 random seeds:         99.1% (110/111)")
print()
print("By cohort (paper Table 5.3):")
print("  MHA (63 pairs): ratio 251x  |  GQA (42): 428x  |  Gemma (6): 863x")

The orange bars (random-direction reduction) are essentially zero across all models and concepts. Projecting out a random direction from a high-dimensional residual stream has no measurable effect on concept separation — the concept direction is not "any direction," it is a specific, geometrically meaningful direction.

The 377× median ratio and 253-sigma z-score leave no ambiguity: the ablation effect is carried by the concept direction specifically. This directly answers the sceptical alternative hypothesis that separation suppression is an artifact of dimensionality reduction rather than directional specificity.

**Why 10 random seeds?** With 10 seeds, the minimum achievable empirical p-value for "concept beats all seeds" is 1/11 ≈ 0.09, which does not formally reach p < 0.05 per pair. The substantive conclusion stands on the ratio magnitude rather than the per-pair p-value.

## 6. Scale analysis (paper §6)

Paper §6 characterises how EEC and probe quality vary across scales. Two key findings:

1. **GEM structure is present at all scales** — even pythia-70m (70M parameters) shows mean EEC < 0.35. Rotation during assembly is not an emergent large-model property.

2. **Probe quality improves with scale** — improvement rates: < 500M = 47% (or 61% excluding gpt2), 500M–3B = 71%, > 3B = 70%.

3. **Non-monotonic EEC in Qwen2.5** — Qwen2.5-1.5B (0.392) and 3B (0.365) show higher EEC than 7B (0.257) and 14B (0.232), suggesting different assembly patterns at mid-scale, not weaker GEM structure.

The cells below show the Pythia scale ladder EEC and Qwen2.5 non-monotonic pattern.

In [ ]:
PYTHIA_LADDER = [
    ("EleutherAI/pythia-70m",   "70M",  70),
    ("EleutherAI/pythia-160m",  "160M", 160),
    ("EleutherAI/pythia-410m",  "410M", 410),
    ("EleutherAI/pythia-1b",    "1B",   1000),
    ("EleutherAI/pythia-1.4b",  "1.4B", 1400),
    ("EleutherAI/pythia-2.8b",  "2.8B", 2800),
    ("EleutherAI/pythia-6.9b",  "6.9B", 6900),
    ("EleutherAI/pythia-12b",   "12B",  12000),
]

QWEN_LADDER = [
    ("Qwen/Qwen2.5-0.5B",  "0.5B",  500),
    ("Qwen/Qwen2.5-1.5B",  "1.5B",  1500),
    ("Qwen/Qwen2.5-3B",    "3B",    3000),
    ("Qwen/Qwen2.5-7B",    "7B",    7000),
    ("Qwen/Qwen2.5-14B",   "14B",   14000),
]

def mean_eec_for_model(model_id: str) -> float | None:
    vals = []
    for concept in ALL_CONCEPTS:
        try:
            caz_data = load_caz(model_id, concept)
        except FileNotFoundError:
            continue
        metrics  = caz_data["layer_data"]["metrics"]
        n_layers = caz_data["n_layers"]
        lm = [LayerMetrics(layer=m["layer"], separation=m["separation_fisher"],
                           coherence=m["coherence"], velocity=m["velocity"])
              for m in metrics]
        profile = find_caz_regions(lm)
        if profile.n_regions == 0:
            continue
        region    = profile.regions[-1]
        entry_vec = np.array(metrics[region.start]["dom_vector"], dtype=np.float64)
        exit_vec  = np.array(metrics[region.end  ]["dom_vector"], dtype=np.float64)
        entry_vec /= (np.linalg.norm(entry_vec) + 1e-12)
        exit_vec  /= (np.linalg.norm(exit_vec)  + 1e-12)
        vals.append(abs(float(np.dot(entry_vec, exit_vec))))
    return float(np.mean(vals)) if vals else None

print("Computing mean EEC for Pythia and Qwen scale ladders...")
pythia_eecs = [(label, params, mean_eec_for_model(mid)) for mid, label, params in PYTHIA_LADDER]
qwen_eecs   = [(label, params, mean_eec_for_model(mid)) for mid, label, params in QWEN_LADDER]
print("Done.")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("white")
fig.suptitle("Mean EEC across scale ladders (lower = more rotation during assembly)",
             fontsize=12, fontweight="bold")

for ax, ladder, title, color in [
    (axes[0], pythia_eecs, "Pythia (MHA)",  "#1565C0"),
    (axes[1], qwen_eecs,   "Qwen2.5 (GQA)", "#2E7D32"),
]:
    labels  = [x[0] for x in ladder]
    eec_vals = [x[2] for x in ladder]
    xs = range(len(labels))
    ax.plot(xs, eec_vals, "-o", color=color, lw=2, ms=8)
    for i, (lbl, _, eec) in enumerate(ladder):
        if eec is not None:
            ax.annotate(f"{eec:.3f}",
                        xy=(i, eec), xytext=(0, 10), textcoords="offset points",
                        ha="center", fontsize=8.5, color=color)
    ax.set_xticks(list(xs))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel("Mean EEC", fontsize=10)
    ax.set_xlabel("Model scale", fontsize=10)
    ax.set_ylim(0, 0.5)
    apply_theme(ax)

plt.tight_layout()
plt.show()

print()
print("Paper per-model mean EEC (Table 2 selective):")
print("  gpt2: 0.083  |  Llama-3.1-8B: 0.087  |  pythia-70m: 0.250")
print("  Qwen2.5-1.5B: 0.392  |  gemma-2-2b: 0.372  (highest EEC = least rotation)")
print()
print("Qwen non-monotonic pattern (paper §6.1):")
print("  1.5B > 3B > 7B > 14B in EEC — higher EEC at mid-scale suggests")
print("  faster settling from a less-differentiated initial direction, not weaker GEM structure.")

### Handoff depth by concept (paper Table 3)

With N=250 pairs, handoff layers cluster later in model depth than in earlier pilot data. Mean handoff relative depth ($L_H / N$) by concept class (paper §4.3):

| Class | Concepts | Mean rel. depth |
|-------|----------|----------------|
| Shallow (< 0.72) | specificity, negation, plurality | < 0.72 |
| Mid (0.72–0.86) | credibility, sentiment, formality, moral_valence, urgency, sarcasm, authorization, agency, exfiltration, causation | 0.72–0.86 |
| Deep (≥ 0.86) | threat_severity, certainty, deception, temporal_order | ≥ 0.86 |

The deep class contains safety-relevant (threat_severity) and epistemic (certainty, deception) concepts — consistent with those concepts requiring more model depth to assemble (§9.2). Notably, certainty and threat_severity rotate *least* (high EEC, §4) yet hand off *latest* — EEC and handoff depth are independent dimensions.

In [ ]:
# Handoff depth per concept for 3 models — shows the deep/mid/shallow ordering
DEPTH_DEMO_MODELS = [
    "EleutherAI/pythia-2.8b",
    "Qwen/Qwen2.5-7B",
    "meta-llama/Llama-3.1-8B",
]

concept_depths = {c: [] for c in ALL_CONCEPTS}
for mid in DEPTH_DEMO_MODELS:
    for concept in ALL_CONCEPTS:
        try:
            caz_data = load_caz(mid, concept)
        except FileNotFoundError:
            continue
        metrics  = caz_data["layer_data"]["metrics"]
        n_layers = caz_data["n_layers"]
        lm = [LayerMetrics(layer=m["layer"], separation=m["separation_fisher"],
                           coherence=m["coherence"], velocity=m["velocity"])
              for m in metrics]
        profile = find_caz_regions(lm)
        if profile.n_regions > 0:
            region = profile.regions[-1]
            L_H = min(region.end + 1, n_layers - 1)
            concept_depths[concept].append(L_H / n_layers)

mean_depths = {c: np.mean(concept_depths[c]) for c in ALL_CONCEPTS if concept_depths[c]}
sorted_c = sorted(mean_depths, key=lambda c: mean_depths[c])

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor("white")

colors_by_class = {
    "specificity": "#1565C0", "negation": "#1565C0", "plurality": "#1565C0",
}
bar_colors = ["#C62828" if mean_depths[c] >= 0.86
              else "#1565C0" if mean_depths[c] < 0.72
              else "#2E7D32" for c in sorted_c]

ax.bar(range(len(sorted_c)), [mean_depths[c] for c in sorted_c],
       color=bar_colors, alpha=0.85)
ax.axhline(0.86, color="#C62828", lw=1.5, ls="--", alpha=0.7, label="Deep threshold (0.86)")
ax.axhline(0.72, color="#1565C0", lw=1.5, ls="--", alpha=0.7, label="Shallow threshold (0.72)")
ax.set_xticks(range(len(sorted_c)))
ax.set_xticklabels([c.replace("_", "\n") for c in sorted_c], fontsize=8)
ax.set_ylabel("Mean L_H / N (3 demo models)", fontsize=10)
ax.set_title("Handoff depth by concept — sorted ascending",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
apply_theme(ax)
plt.tight_layout()
plt.show()

print("Paper (23 models): median L_H/N >= 0.917 for 14/17 concepts.")
print("specificity (median 0.583) and plurality (median 0.656) are consistently early.")

## 7. Failure modes: gpt2 and OPT-6.7b (paper §7)

Two models show anomalous behaviour; they have different causes and different resolutions.

### 7.1 gpt2 — genuine structured failure

In 13/17 concept × model pairs, ablation at the gpt2 handoff layer pathologically *increases* concept separation (retained percentage > 100%) instead of suppressing it. The root cause is architectural: gpt2 has only 12 layers. Its handoff typically falls at layer 11 (HL/N ≈ 0.92), where ablation engages unembedding-preparation computations rather than concept directions — removing the concept direction at that depth releases separation rather than suppressing it.

gpt2 is a genuine, replicating failure mode. It is excluded from the MHA/GQA cohort comparison and is the primary motivation for the depth-corrected near-final rule (N ≥ 20).

### 7.2 OPT-6.7b — corpus-size artifact

At N < 250, OPT-6.7b showed only 7/17 = 41% handoff improvement — below chance. This was initially interpreted as a structured failure from high EEC (concepts converging before the handoff). At N=250, this reverses: OPT-6.7b achieves 13/17 = 76% — above the corpus mean. The initial finding was corpus-size-dependent noise, not an architectural invariant.

In [ ]:
# Demonstrate gpt2 pathological ablation vs OPT normal ablation
FAILURE_MODELS = [
    ("openai-community/gpt2",   "gpt2 (failure)",   True),
    ("facebook/opt-6.7b",       "OPT-6.7b (normal)", False),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
fig.patch.set_facecolor("white")
fig.suptitle("gpt2 ablation failure vs OPT-6.7b recovery at N=250",
             fontsize=12, fontweight="bold")

for ax, (mid, label, is_failure) in zip(axes, FAILURE_MODELS):
    handoff_vals, peak_vals, concept_labels = [], [], []
    pathological = []

    for concept in ALL_CONCEPTS:
        try:
            abl = load_ablation_gem(mid, concept)
        except FileNotFoundError:
            continue
        comp = abl["comparison"]
        handoff_vals.append(comp["handoff_retained_pct"])
        peak_vals.append(comp["peak_retained_pct"])
        concept_labels.append(concept.replace("_", "\n"))
        if comp["handoff_retained_pct"] > 100:
            pathological.append(concept)

    xs = range(len(concept_labels))
    bar_color = ["#C62828" if h > 100 else "#1565C0" for h in handoff_vals]
    ax.bar([x - 0.2 for x in xs], handoff_vals,
           width=0.38, label="Handoff retained %",
           color=bar_color, alpha=0.85)
    ax.bar([x + 0.2 for x in xs], peak_vals,
           width=0.38, label="Peak retained %",
           color="#E65100", alpha=0.75)
    ax.axhline(100, color="black", lw=1.5, ls="--", alpha=0.6, label="100% (no suppression)")

    ax.set_xticks(list(xs))
    ax.set_xticklabels(concept_labels, fontsize=7)
    ax.set_ylabel("Retained %", fontsize=10)
    n_better = sum(1 for h, p in zip(handoff_vals, peak_vals) if h <= p)
    ax.set_title(f"{label}\n{n_better}/{len(handoff_vals)} handoff better  "
                 f"| {len(pathological)} pathological (>100%)",
                 fontsize=10, fontweight="bold")
    ax.legend(fontsize=8)
    apply_theme(ax)

plt.tight_layout()
plt.show()

print("gpt2 (12 layers):")
print("  13/17 concepts: handoff ablation INCREASES separation (ablation overshoot)")
print("  Root cause: L_H at layer 11 (91.7% depth) — unembedding-preparation zone")
print("  Replicates across corpus sizes — genuine architectural failure mode")
print()
print("OPT-6.7b (32 layers):")
print("  N < 250: 7/17 = 41% improvement (below chance, initially classified as failure)")
print("  N = 250: 13/17 = 76% improvement (above corpus mean)")
print("  Reversal shows: the small-corpus finding was a statistical artifact, not an architectural invariant")
print("  OPT's high-EEC structural profile (sarcasm: 0.807, temporal_order: 0.627) is real")
print("  but does not prevent GEM from finding productive handoffs at N=250")

The gpt2 case is methodologically important: it shows that GEM's handoff detection is sensitive to the unembedding layer's proximity, and that very shallow models (N < 20) require the depth-corrected near-final rule or should be excluded from handoff analysis altogether.

The OPT case is also methodologically important in the opposite direction: it shows that initial below-chance results at small N can reverse at N=250, and that high EEC (less rotation during assembly) does not necessarily prevent GEM from finding useful handoff layers. The lesson is not to classify a model as a failure mode until the full corpus has been run.

From the paper (§9.3): "A result that would falsify GEM on rotation-present architectures would be systematic failure even where EEC is low — that pattern is not observed in our corpus."

## Summary

| Section | Paper §§ | Key claim | Status in this notebook |
|---------|----------|-----------|------------------------|
| 1. EEC and handoff formalism | §3 | EEC computed from dom_vectors; L_H = CAZ_end + 1 | Verified for 3 pairs; API shown |
| 2. EEC distribution | §4, Table 1 | Mean 0.233; 93.9% < 0.5; 23.8% near-orthogonal | Computed for 3 models; paper aggregate stated |
| 3. Handoff vs peak validation | §5.1 | 259/391 (66.2%) prefer handoff; p=3.21e-17 | Computed for 3 models; paper aggregate stated |
| 4. Adaptive width rule | §5.2 | Near-final rule improves 62/89 triggered cases (+8.51pp) | Trigger logic shown; counts stated |
| 5. Direction-specificity control | §5.3 | 377× median ratio vs random ablation | Computed for 3 models; paper aggregate stated |
| 6. Scale analysis | §6 | GEM present at all scales; Qwen non-monotonic EEC | Pythia and Qwen ladders computed |
| 7. Failure modes | §7 | gpt2 genuine failure; OPT corpus-size artifact | Both plotted and explained |

---

### Reproducing the paper aggregate results

All aggregate statistics (391 pairs, 23 models) can be reproduced by running each section's demo code over `PRIMARY_MODELS` instead of the 3-model demo subset. The data lives at `~/rosetta_data/paper_n250/` — no extraction required.

### Papers and resources

| | |
|-|-|
| Paper 1 — CAZ Framework | Henry (2026a) |
| Paper 2 — GEM | Henry (2026b) |
| Paper 3 — CAZ Validation | Henry (2026c) |
| Paper 4 — PRH | Henry (2026d) |
| `rosetta_tools` | [GitHub](https://github.com/jamesrahenry/Rosetta_Tools) |
| Concept pairs dataset | [Rosetta_Concept_Pairs](https://github.com/jamesrahenry/Rosetta_Concept_Pairs) |
| Activations dataset | [Rosetta-Activations](https://huggingface.co/datasets/james-ra-henry/Rosetta-Activations) |